In [20]:
import os
import sys
import pandas as pd
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

Настройки моделей и параметров

In [21]:
SUMY_LANG = 'russian' # Язык для Sumy (LexRank)
T5_MODEL = 'IlyaGusev/rut5_base_sum_gazeta' # Название предобученной T5-модели для суммаризации

LexRank — это графовый алгоритм извлечения ключевых предложений, основанный на идее PageRank. Он строит граф предложений, где ребра — это семантическая схожесть, и выбирает центральные узлы.

In [22]:
# Функция summarize_lexrank 
def summarize_lexrank(text: str, n_sentences: int) -> str:
    """
    Выполняет извлечение ключевых предложений с помощью LexRank.
    :param text: исходный текст.
    :param n_sentences: число предложений в итоговой суммаризации.
    :return: строка из n_sentences ключевых предложений.
    """
    # Парсим текст и токенизируем
    parser = PlaintextParser.from_string(text, Tokenizer(SUMY_LANG))
    # Инициализируем Summarizer
    summarizer = LexRankSummarizer()
    # Получаем список предложений
    summary = summarizer(parser.document, n_sentences)
    # Объединяем предложения в одну строку
    return ' '.join(str(s) for s in summary)


T5 — это трансформерная модель генерации текста, которая обучена на задаче преобразования одного текста в другой.

In [23]:
# Инициализация T5-модели один раз
_t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL)
_t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)
# Функция summarize_t5 
def summarize_t5(text: str, min_len: int, max_len: int, beams: int = 4) -> str:
    """
    Генерирует суммаризацию с помощью T5-модели.
    :param text: исходный текст.
    :param min_len: минимальная длина результирующего текста (токены).
    :param max_len: максимальная длина результирующего текста (токены).
    :param beams: число лучей для beam search.
    :return: сгенерированная суммаризация.
    """
    # Токенизация входного текста
    inputs = _t5_tokenizer(
        text, 
        max_length=512, # max_length: обрезаем вход до 512 токенов
        truncation=True,         # truncation: включить усечение, если текст длиннее max_length
        return_tensors='pt'         # return_tensors: формат выходных тензоров ('pt' для PyTorch)
    )
    # Генерация резюме с beam search и ограничением на повтор n-грамм
    out = _t5_model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_length=max_len, # максимальная длина генерируемого текста
        min_length=min_len, # минимальная длина генерируемого текста
        num_beams=beams, # ширина beam search
        no_repeat_ngram_size=3 # запрет повтора трехграмм
    )
    # Декодируем и удаляем специальные токены
    return _t5_tokenizer.decode(out[0], skip_special_tokens=True)

In [24]:
if __name__ == '__main__':
    # Детектируем запуск в Jupyter или как скрипт
    if 'ipykernel' in sys.argv[0] or 'PYCHARM_HOSTED' in os.environ:
        input_path = 'Bakhvalov_IMS_2020_rus.txt'
        output_csv = 'summaries.csv'
    else:
        import argparse
        parser = argparse.ArgumentParser(description='Суммаризация текстового файла')
        parser.add_argument('input_path', help='Путь к .txt файлу')
        parser.add_argument('-o', '--output_csv', default='summaries.csv',
                            help='Имя CSV с результатами')
        args = parser.parse_args()
        input_path = args.input_path
        output_csv = args.output_csv

    # Проверка
    if not os.path.isfile(input_path) or not input_path.lower().endswith('.txt'):
        print(f"Error: файл '{input_path}' не найден или не является .txt")
        sys.exit(1)

    # Читаем текст
    with open(input_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read().strip()

In [25]:
    # Список словарей с результатами
    summaries = []
    # LexRank
    summaries.append({'method': 'LexRank', 'type': 'Заголовок', 'summary': summarize_lexrank(text, 1)})
    summaries.append({'method': 'LexRank', 'type': 'Аннотация', 'summary': summarize_lexrank(text, 3)})
    summaries.append({'method': 'LexRank', 'type': 'Реферат',   'summary': summarize_lexrank(text, 5)})
    # T5
    summaries.append({'method': 'T5',      'type': 'Заголовок', 'summary': summarize_t5(text, 5, 15)})
    summaries.append({'method': 'T5',      'type': 'Аннотация', 'summary': summarize_t5(text, 30, 60)})
    summaries.append({'method': 'T5',      'type': 'Пересказ',  'summary': summarize_t5(text, 60, 150)})


In [26]:
    # Сохраняем
    df = pd.DataFrame(summaries)
    df.to_csv(output_csv, index=False)

In [27]:
    # Печать
    print('LexRank:')
    for item in df.query("method=='LexRank'").itertuples():
        print(f"- {item.type}: {item.summary}")
    print('\nT5:')
    for item in df.query("method=='T5'").itertuples():
        print(f"- {item.type}: {item.summary}")

LexRank:
- Заголовок: Такие же графы были сгенерированы для всего набора данных и для троек, включающих ошибку из набора данных.
- Аннотация: Тем самым мы предполагаем, что в категории ошибок нет ни одной другой ошибки кроме как из данной категории. Процесс генерации правил выглядит следующим образом. Такие же графы были сгенерированы для всего набора данных и для троек, включающих ошибку из набора данных.
- Реферат: Поэтому задача генерации правил для ошибок из категории семантики не имеет смысла. Тем самым мы предполагаем, что в категории ошибок нет ни одной другой ошибки кроме как из данной категории. Выделение признаков После разделения данных по ошибкам из слов и предложения были выделены признаки (см. табл. 3) в качестве основы для генерации правил. Процесс генерации правил выглядит следующим образом. Такие же графы были сгенерированы для всего набора данных и для троек, включающих ошибку из набора данных.

T5:
- Заголовок: Автоматическая проверка правописания – это задача автома